# Άνθρωπος-στο-Βρόχο: Πύλες Προ-Δράσης, Κατηγοριοποίηση Κινδύνου, και Καταγραφή Ελέγχου

Το README για αυτό το μάθημα εισάγει το Άνθρωπος-στο-Βρόχο με ένα σύντομο απόσπασμα που ζητά από τον χρήστη να `ΕΓΚΡΙΝΕΙ` ή να `ΑΠΟΡΡΙΨΕΙ` αφού ο πράκτορας έχει ήδη παράγει μια απάντηση. Αυτό το μοτίβο είναι ένα καλό σημείο εκκίνησης, αλλά οι υλοποιήσεις HITL σε παραγωγή συνήθως χρειάζονται τρία επιπλέον στοιχεία:

1. Μια **πύλη προ-δράσης** που λειτουργεί **πριν** ο πράκτορας εκτελέσει ένα επικίνδυνο βήμα, ώστε το κόστος, η μη αναστρεψιμότητα και η καθυστέρηση να παραμένουν υπό έλεγχο.
2. **Κατηγοριοποίηση κινδύνου**, ώστε οι ενέργειες χαμηλού κινδύνου να εκτελούνται αυτόματα, οι ενέργειες μέσου κινδύνου να εγκρίνονται κατά παρτίδες, και μόνο οι ενέργειες υψηλού κινδύνου να μπλοκάρουν ώστε να απαιτείται άνθρωπος.
3. Ένα **ημερολόγιο ελέγχου μαζί με βρόχο αναθεώρησης**, ώστε κάθε απόφαση πύλης να καταγράφεται ως JSONL, και μια απόρριψη να επαναπροτρέπει τον πράκτορα με ένα δομημένο λόγο αντί απλώς να εμφανίζει `Αναθεώρηση...`.

Αυτό το σημειωματάριο χτίζει το καθένα από αυτά πάνω στις ίδιες βασικές λειτουργίες όπως το `06-system-message-framework.ipynb`. Εκτελείται από άκρη σε άκρη σε `DEMO_MODE = True` (χωρίς ανάγκη διαδραστικής εισόδου) ή με πραγματικές προτροπές `input()` όταν `DEMO_MODE = False`. Σημείωση: στο DEMO_MODE η επανάληψη του τρίτου στόχου είναι σε σενάριο ώστε οι μηχανισμοί του βρόχου να είναι ορατοί ολόκληροι. Η πραγματική αναθεωρημένη επαν-κατηγοριοποίηση απαιτεί `DEMO_MODE = False` και έναν χειριστή.

**Εκτός πεδίου (αντιμετωπίζεται σε άλλα μαθήματα):** αυθεντικοποίηση και έλεγχος πρόσβασης (Lesson 06 README threat 2), ενδιάμεσο λογισμικό για κλήση εργαλείων (Lesson 14 MAF deep dive), μοτίβα πολυ-πράκτορα αντιδικίας.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Πρότυπο 1: Πύλη πριν από τη δράση

Το απόσπασμα HITL στο README καλεί πρώτα τον πράκτορα και μετά ζητά από τον χρήστη να εγκρίνει το αποτέλεσμα. Αυτό είναι μια ροή **μετά τη δράση**. Ο πράκτορας έχει ήδη εκτελέσει, οπότε το κόστος κλήσης του LLM έχει ήδη πληρωθεί, και οποιαδήποτε παρενέργεια (αποστολή email, εγγραφή σε βάση δεδομένων, ανάρτηση σχολίου) έχει ήδη συμβεί.

Μια ροή **προ δράσης** εισάγει την πύλη πριν ο πράκτορας εκτελέσει το ριψοκίνδυνο βήμα. Ο πράκτορας προτείνει τη δράση, η πύλη αποφασίζει αν θα εκτελεστεί, και μόνο με την έγκριση συμβαίνει η παρενέργεια.

| Πτυχή | Έγκριση μετά τη δράση (απόσπασμα README) | Πύλη πριν τη δράση (αυτό το σημειωματάριο) |
|---|---|---|
| Πότε εκτελείται η έγκριση; | Μετά την παραγωγή αποτελέσματος του πράκτορα | Πριν εκτελεστεί οποιαδήποτε παρενέργεια |
| Κόστος LLM σε απόρριψη | Έχει ήδη πληρωθεί | Πληρώνεται μόνο για την πρόταση, όχι για τη δράση |
| Αντίστροφες παρενέργειες | Πιθανές (η δράση έχει ήδη συμβεί) | Αποτρέπονται |
| Διαύγεια ελέγχου | Η έγκριση είναι μια εντολή εκτύπωσης | Η έγκριση είναι εγγραφή JSON με χρονική σήμανση, δράση, λόγο |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Πρότυπο 2: Κατηγοριοποίηση κινδύνου

Δεν χρειάζεται κάθε ενέργεια να έχει έγκριση από άνθρωπο. Μια αναζήτηση μόνο για ανάγνωση μέσω δημόσιου API έχει διαφορετικούς κινδύνους σε σχέση με την αποστολή email σε πελάτη. Η ισοπέδωση και των δύο σπαταλά την προσοχή του χειριστή και επιβραδύνει τον πράκτορα.

Ένα απλό μοντέλο 3 επιπέδων:

| Επίπεδο | Παραδείγματα | Ροή έγκρισης |
|---|---|---|
| `χαμηλό` (μόνο ανάγνωση) | Αναζήτηση σε βάση γνώσεων, αναζήτηση επιλογών πτήσεων, ανάκτηση δημόσιας ιστοσελίδας | Αυτόματη εκτέλεση, καταγραφή για έλεγχο |
| `μεσαίο` (φθηνή μεταβολή) | Αποθήκευση αποτελέσματος, αύξηση μετρητή, προγραμματισμός υπενθύμισης | Αυτόματη εκτέλεση, αλλά με καθημερινή συνολική ανασκόπηση |
| `υψηλό` (εξωτερική πρόσβαση ή μη αναστρέψιμο) | Αποστολή email, χρέωση κάρτας, δημοσίευση σε δημόσιο κανάλι | Μπλοκάρισμα μέχρι τη χειροκίνητη έγκριση |

Αυτή είναι μια κατηγοριοποίηση. Τα συστήματα παραγωγής χρησιμοποιούν συχνά πιο λεπτομερή επίπεδα (π.χ., επίπεδα δικαιωμάτων AWS IAM, ρόλοι βασισμένοι σε επίπεδα πρόσβασης). Η τριπλή κατηγοριοποίηση παρακάτω είναι η πιο απλή και χρήσιμη εκδοχή για έναν πράκτορα που αναμειγνύει ενέργειες μόνο για ανάγνωση και ενέργειες με παρενέργειες.

Ο ταξινομητής παρακάτω χρησιμοποιεί ευρετικούς κανόνες βασισμένους σε λέξεις-κλειδιά ώστε το demo να παραμένει ντετερμινιστικό και φθηνό. Σε σύστημα παραγωγής θα αντικαθιστούσατε αυτό με έναν εκπαιδευμένο ταξινομητή ή έναν μηχανισμό πολιτικής.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Πρότυπο 3: Καταγραφή ελέγχου και βρόχος αναθεώρησης

Ένα `print("Response approved.")` δεν είναι καταγραφή ελέγχου. Για αξιοπιστία, κάθε απόφαση πύλης πρέπει να καταγράφεται ως δομημένο γεγονός που μπορείτε αργότερα να ερωτήσετε, να αναπαράγετε ή να επισυνάψετε σε ανασκόπηση συμβάντος.

Δύο μέρη:

1. **Μόνο προσθήκη JSONL.** Μια γραμμή ανά απόφαση, με χρονική σήμανση, ενέργεια, επίπεδο, απόφαση, λόγο. Εύκολο για grep, εύκολο για αποστολή σε πραγματική αποθήκη καταγραφής αργότερα.
2. **Βρόχος αναθεώρησης κατά την απόρριψη.** Όταν η πύλη επιστρέφει `deny`, ο πράκτορας επανυποβάλλει εαυτόν με τον λόγο απόρριψης στο πλαίσιο, ώστε η επόμενη πρόταση να αποφύγει το πρόβλημα.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Πρόσθετοι πόροι

Πολλά άλλα δημόσια έργα υλοποιούν παραλλαγές αυτών των προτύπων HITL. Συγκρίνετε προσεγγίσεις για να βρείτε τι ταιριάζει στο stack σας:

- **LangChain** ανθρώπινα-στη-βουρτσά εργαλεία wrapper ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): εργαλεία wrapper που σταματούν την εκτέλεση για ανθρώπινη είσοδο.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); Το AutoGen v0.4+ αναδιαρθρώθηκε): χρησιμοποιεί έναν ρόλο πράκτορα ειδικά για να εκπροσωπήσει τον άνθρωπο σε συνομιλίες πολλαπλών πρακτόρων.
- **Microsoft Agent Framework (MAF)** ενδιάμεσο λογισμικό κλήσης λειτουργιών ([docs](https://learn.microsoft.com/agent-framework/)): ενδιάμεσο λογισμικό που εκτελείται γύρω από κάθε κλήση εργαλείου/λειτουργίας, κατάλληλο για λογική εγκρίσεων και ροές έγκρισης.

Κάθε έργο χειρίζεται τις τρεις υπο-μορφές διαφορετικά: το LangChain τις τυλίγει ως εργαλεία, το AutoGen χρησιμοποιεί ρόλο πράκτορα, και το Microsoft Agent Framework χρησιμοποιεί ενδιάμεσο λογισμικό κλήσης λειτουργιών. Διαβάστε μία ή δύο υλοποιήσεις από την αρχή ως το τέλος πριν επιλέξετε ένα σχεδιασμό για τον δικό σας πράκτορα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
